# Exploratory Data Analysis — B2B Transaction Data

**Dataset:** `B2B_Transaction_Data_.xlsx` (sheet `Transaction`) — line-level B2B sales of a Turkish book / puzzle distributor, calendar year 2020.

| Column | Meaning |
|---|---|
| `InvoiceNo` | Invoice ID; first 3 letters = document/branch prefix (BAD, BSR, DPF, DPE, MUH) |
| `CustomerID` | B2B customer (store / reseller) |
| `InvoiceDate` | Invoice date |
| `StockCode` | Product barcode (EAN/ISBN) |
| `Description` | Product name |
| `Category` | Product category (Turkish) |
| `City` | Customer city |
| `Quantity` | Units sold on the line |
| `NetPrice` | Net (discounted) unit price paid by the customer |
| `UnitPrice` | List unit price |

**Outline**
1. Setup & loading
2. Structure, types, missing values, duplicates
3. Data quality issues & cleaning
4. Feature engineering (revenue, discount, time parts)
5. Univariate analysis
6. Time-series analysis
7. Customers (incl. Pareto & RFM preview)
8. Products & categories
9. Geography
10. Invoice prefixes / channels
11. Correlations
12. Key findings

All charts use **Plotly** (interactive: hover for exact values, drag to zoom, double-click to reset).

## 1. Setup & loading

In [ ]:
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

# Works in Jupyter Notebook/Lab and VS Code
pio.renderers.default = "plotly_mimetype+notebook_connected"

# Plot style: recessive grid, one accent hue for single-series charts
BLUE, GREY = "#2a78d6", "#a3a29d"
pio.templates["eda"] = go.layout.Template(layout=dict(
    font=dict(family="Segoe UI, Arial, sans-serif", size=12, color="#52514e"),
    title=dict(font=dict(size=15, color="#0b0b0b")),
    paper_bgcolor="white", plot_bgcolor="white",
    xaxis=dict(gridcolor="#ecebe7", linecolor="#c9c8c3", zeroline=False),
    yaxis=dict(gridcolor="#ecebe7", linecolor="#c9c8c3", zeroline=False),
    colorway=[BLUE, "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"],
    margin=dict(l=60, r=30, t=60, b=50), bargap=0.2,
    hoverlabel=dict(bgcolor="white", font_size=12),
))
pio.templates.default = "plotly_white+eda"

In [ ]:
%%time
df_raw = pd.read_excel("B2B_Transaction_Data_.xlsx", sheet_name="Transaction")
df = df_raw.copy()
df.head()

## 2. Structure, types, missing values, duplicates

In [ ]:
print(f"Rows: {df.shape[0]:,}   Columns: {df.shape[1]}")
df.info(memory_usage="deep")

In [ ]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(3),
    "unique": df.nunique(),
})
summary

In [ ]:
df.describe().T

In [ ]:
n_dup = df.duplicated().sum()
print(f"Fully duplicated rows: {n_dup:,} ({n_dup/len(df):.2%})")
df[df.duplicated(keep=False)].sort_values(["InvoiceNo", "StockCode"]).head(6)

> Exact duplicate lines are plausible in B2B invoicing (the same item scanned twice on one invoice), but there is no line-number column to tell a real second line from a double entry, so the dashboard **removes** them and keeps the first copy.

## 3. Data quality issues & cleaning

In [ ]:
# Description has mixed types (Excel converted some product names to numbers / dates)
print(df["Description"].map(type).value_counts())
df["Description"] = df["Description"].astype(str).str.strip()

# Dates arrive as strings
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
print("Date range:", df["InvoiceDate"].min().date(), "->", df["InvoiceDate"].max().date())

# StockCode is an identifier, not a number
df["StockCode"] = df["StockCode"].astype(str)

In [ ]:
issues = {
    "Quantity missing":        df["Quantity"].isna().sum(),
    "Quantity == 0":           (df["Quantity"] == 0).sum(),
    "NetPrice < 0":            (df["NetPrice"] < 0).sum(),
    "NetPrice == 0":           (df["NetPrice"] == 0).sum(),
    "NetPrice > UnitPrice":    (df["NetPrice"] > df["UnitPrice"]).sum(),
    "UnitPrice < 0.01":        (df["UnitPrice"] < 0.01).sum(),
    "City missing":            df["City"].isna().sum(),
    "Category missing":        df["Category"].isna().sum(),
}
pd.Series(issues, name="rows").to_frame()

In [ ]:
# Inspect the anomalous rows
df[(df["NetPrice"] <= 0) | (df["NetPrice"] > df["UnitPrice"]) | (df["Quantity"].isna()) | (df["Quantity"] == 0)]

In [ ]:
# Missing values per column (visual)
miss = df.isna().sum().sort_values()
fig = px.bar(miss, orientation="h", title="Missing values per column", labels={"value": "Missing rows", "index": ""})
fig.update_traces(marker_color=BLUE, hovertemplate="%{y}: %{x:,}<extra></extra>")
fig.update_layout(showlegend=False, height=350)
fig.show()

In [ ]:
# Cleaning decisions (a handful of rows -> drop rather than impute)
before = len(df)
df = df[df["Quantity"].notna() & (df["Quantity"] > 0)]
df = df[(df["NetPrice"] > 0) & (df["UnitPrice"] >= 0.01)]
df["City"] = df["City"].fillna("UNKNOWN")
df["Category"] = df["Category"].fillna("UNKNOWN")
print(f"Removed {before - len(df):,} rows -> {len(df):,} remain")

## 4. Feature engineering

In [ ]:
df["Revenue"]     = df["Quantity"] * df["NetPrice"]           # what the customer paid
df["ListValue"]   = df["Quantity"] * df["UnitPrice"]          # value at list price
df["DiscountPct"] = 1 - df["NetPrice"] / df["UnitPrice"]     # trade discount
df["Prefix"]      = df["InvoiceNo"].str[:3]
df["Month"]       = df["InvoiceDate"].dt.to_period("M").dt.to_timestamp()
df["Weekday"]     = df["InvoiceDate"].dt.day_name()
df["Quarter"]     = df["InvoiceDate"].dt.quarter

kpi = pd.Series({
    "Total revenue (net)":   df["Revenue"].sum(),
    "Total list value":      df["ListValue"].sum(),
    "Effective discount":    1 - df["Revenue"].sum() / df["ListValue"].sum(),
    "Units sold":            df["Quantity"].sum(),
    "Invoices":              df["InvoiceNo"].nunique(),
    "Customers":             df["CustomerID"].nunique(),
    "Products (StockCode)":  df["StockCode"].nunique(),
    "Categories":            df["Category"].nunique(),
    "Cities":                df["City"].nunique(),
    "Avg revenue / invoice": df.groupby("InvoiceNo")["Revenue"].sum().mean(),
    "Avg lines / invoice":   df.groupby("InvoiceNo").size().mean(),
})
kpi.to_frame("value").style.format("{:,.2f}")

## 5. Univariate analysis

In [ ]:
# Distributions on log10 scale (values are heavily right-skewed)
cols = ["Quantity", "NetPrice", "UnitPrice", "Revenue"]
fig = make_subplots(rows=2, cols=2, subplot_titles=[f"log10({c})" for c in cols])
for i, c in enumerate(cols):
    fig.add_trace(go.Histogram(x=np.log10(df[c]), nbinsx=60, marker_color=BLUE, name=c,
                               hovertemplate="log10=%{x}<br>lines=%{y:,}<extra></extra>"),
                  row=i // 2 + 1, col=i % 2 + 1)
fig.update_layout(title="Distributions of numeric columns", showlegend=False, height=600, bargap=0.05)
fig.show()

In [ ]:
q = df[["Quantity", "NetPrice", "UnitPrice", "Revenue", "DiscountPct"]].quantile([.01, .05, .25, .5, .75, .95, .99, 1]).T
q

In [ ]:
# Outliers by IQR rule
def iqr_outliers(s):
    q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    return ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).mean()
pd.Series({c: iqr_outliers(df[c]) for c in ["Quantity", "NetPrice", "UnitPrice", "Revenue"]}, name="share_outliers").map("{:.1%}".format).to_frame()

In [ ]:
fig = px.histogram(df, x=df["DiscountPct"].clip(0, 0.8), nbins=80, title="Trade discount per line (NetPrice vs UnitPrice)")
fig.update_traces(marker_color=BLUE, hovertemplate="discount=%{x:.0%}<br>lines=%{y:,}<extra></extra>")
fig.update_layout(xaxis_title="Discount", yaxis_title="Lines", xaxis_tickformat=".0%", height=380, bargap=0.05)
fig.show()

## 6. Time-series analysis

In [ ]:
daily = df.groupby("InvoiceDate").agg(Revenue=("Revenue", "sum"), Invoices=("InvoiceNo", "nunique")).reset_index()
daily["MA7"] = daily["Revenue"].rolling(7, min_periods=1).mean()
fig = go.Figure()
fig.add_scatter(x=daily["InvoiceDate"], y=daily["Revenue"], mode="lines", name="Daily",
                line=dict(color=BLUE, width=1), opacity=.4)
fig.add_scatter(x=daily["InvoiceDate"], y=daily["MA7"], mode="lines", name="7-day mean", line=dict(color=BLUE, width=2.5))
fig.update_layout(title="Daily revenue, 2020", hovermode="x unified", height=420, yaxis_title="Revenue",
                  legend=dict(orientation="h", y=1.08, x=1, xanchor="right"))
fig.update_xaxes(rangeslider_visible=True)
fig.show()

> Note the gap with almost no transactions between late July and early August, and the COVID-19 lockdowns in Turkey from mid-March.

In [ ]:
monthly = df.groupby("Month").agg(
    Revenue=("Revenue", "sum"), Units=("Quantity", "sum"),
    Invoices=("InvoiceNo", "nunique"), Customers=("CustomerID", "nunique"))
monthly["MoM_growth"] = monthly["Revenue"].pct_change()
monthly

In [ ]:
m = monthly.reset_index()
fig = make_subplots(rows=1, cols=2, subplot_titles=["Monthly revenue", "Active customers per month"])
fig.add_bar(x=m["Month"], y=m["Revenue"], marker_color=BLUE, row=1, col=1,
            customdata=m["MoM_growth"], hovertemplate="%{x|%b %Y}<br>Revenue: %{y:,.0f}<br>MoM: %{customdata:+.1%}<extra></extra>")
fig.add_bar(x=m["Month"], y=m["Customers"], marker_color=BLUE, row=1, col=2,
            hovertemplate="%{x|%b %Y}<br>Customers: %{y}<extra></extra>")
fig.update_xaxes(dtick="M1", tickformat="%b")
fig.update_layout(showlegend=False, height=400)
fig.show()

In [ ]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
wd = df.groupby("Weekday").agg(Revenue=("Revenue", "sum"), Days=("InvoiceDate", "nunique")).reindex(order)
wd["Revenue_per_active_day"] = wd["Revenue"] / wd["Days"]
fig = px.bar(wd, x=wd.index, y="Revenue_per_active_day", title="Average revenue per trading day, by weekday",
             labels={"Revenue_per_active_day": "Revenue / day", "Weekday": ""})
fig.update_traces(marker_color=BLUE, hovertemplate="%{x}: %{y:,.0f}<extra></extra>")
fig.update_layout(height=380)
fig.show()
wd

## 7. Customers

In [ ]:
cust = df.groupby("CustomerID").agg(
    Revenue=("Revenue", "sum"), Units=("Quantity", "sum"), Invoices=("InvoiceNo", "nunique"),
    Products=("StockCode", "nunique"), City=("City", lambda s: s.mode().iat[0]),
    FirstBuy=("InvoiceDate", "min"), LastBuy=("InvoiceDate", "max"),
).sort_values("Revenue", ascending=False)
cust["AvgInvoice"] = cust["Revenue"] / cust["Invoices"]
cust.describe().T

In [ ]:
cust.head(15)

In [ ]:
# Pareto: how concentrated is revenue?
cum = cust["Revenue"].cumsum() / cust["Revenue"].sum()
share_cust = np.arange(1, len(cum) + 1) / len(cum)
top20 = cum.iloc[int(len(cum) * .2) - 1]
n80 = (cum < .8).sum() + 1
print(f"Top 20% of customers -> {top20:.1%} of revenue")
print(f"{n80} customers ({n80/len(cum):.1%}) generate 80% of revenue")

fig = go.Figure()
fig.add_scatter(x=share_cust, y=cum.values, mode="lines", line=dict(color=BLUE, width=2.5), name="Customers",
                hovertemplate="Top %{x:.1%} of customers<br>%{y:.1%} of revenue<extra></extra>")
fig.add_scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(color=GREY, dash="dash", width=1), name="Perfect equality", hoverinfo="skip")
fig.add_hline(y=.8, line=dict(color=GREY, width=1)); fig.add_vline(x=n80 / len(cum), line=dict(color=GREY, width=1))
fig.add_annotation(x=n80 / len(cum), y=.8, text=f"{n80} customers = 80% of revenue", showarrow=True, ax=90, ay=40)
fig.update_layout(title="Customer revenue concentration (Pareto curve)", xaxis_title="Share of customers (ranked by revenue)",
                  yaxis_title="Cumulative share of revenue", xaxis_tickformat=".0%", yaxis_tickformat=".0%",
                  height=480, width=700, showlegend=False)
fig.show()

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["log10(Revenue per customer)", "log10(Invoices per customer)"])
fig.add_histogram(x=np.log10(cust["Revenue"]), nbinsx=50, marker_color=BLUE, row=1, col=1)
fig.add_histogram(x=np.log10(cust["Invoices"]), nbinsx=50, marker_color=BLUE, row=1, col=2)
fig.update_layout(showlegend=False, height=380, bargap=0.05, yaxis_title="Customers")
fig.show()

In [ ]:
# RFM preview (snapshot = day after last invoice)
snapshot = df["InvoiceDate"].max() + pd.Timedelta(days=1)
rfm = pd.DataFrame({
    "Recency":   (snapshot - cust["LastBuy"]).dt.days,
    "Frequency": cust["Invoices"],
    "Monetary":  cust["Revenue"],
})
for c, lab in [("Recency", [4, 3, 2, 1]), ("Frequency", [1, 2, 3, 4]), ("Monetary", [1, 2, 3, 4])]:
    rfm[c[0]] = pd.qcut(rfm[c].rank(method="first"), 4, labels=lab).astype(int)
rfm["RFM_score"] = rfm[["R", "F", "M"]].sum(axis=1)
def segment(r):
    if r.R >= 3 and r.F >= 3 and r.M >= 3: return "Champions"
    if r.F >= 3 and r.M >= 3:              return "Loyal, slipping"
    if r.R >= 3 and r.F <= 2:              return "Recent / developing"
    if r.R <= 2 and r.F <= 2:              return "At risk / lost"
    return "Mid-value"
rfm["Segment"] = rfm.apply(segment, axis=1)
seg = rfm.groupby("Segment").agg(Customers=("Monetary", "size"), Revenue=("Monetary", "sum"), AvgRecency=("Recency", "mean"))
seg["Revenue_share"] = seg["Revenue"] / seg["Revenue"].sum()
seg = seg.sort_values("Revenue", ascending=False)
seg

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Customers per segment", "Revenue share per segment"])
s = seg.iloc[::-1]
fig.add_bar(y=s.index, x=s["Customers"], orientation="h", marker_color=BLUE, row=1, col=1,
            hovertemplate="%{y}: %{x} customers<extra></extra>")
fig.add_bar(y=s.index, x=s["Revenue_share"], orientation="h", marker_color=BLUE, row=1, col=2,
            text=s["Revenue_share"].map("{:.1%}".format), textposition="outside",
            hovertemplate="%{y}: %{x:.1%} of revenue<extra></extra>")
fig.update_xaxes(tickformat=".0%", row=1, col=2)
fig.update_yaxes(showticklabels=False, row=1, col=2)
fig.update_layout(title="RFM segments (preview)", showlegend=False, height=380)
fig.show()

## 8. Products & categories

In [ ]:
prod = df.groupby(["StockCode", "Description"]).agg(
    Revenue=("Revenue", "sum"), Units=("Quantity", "sum"), Customers=("CustomerID", "nunique")
).sort_values("Revenue", ascending=False)
print(f"Products: {len(prod):,};  sold only once: {(df.groupby('StockCode').size() == 1).sum():,}")
prod.head(15)

In [ ]:
cat = df.groupby("Category").agg(
    Revenue=("Revenue", "sum"), Units=("Quantity", "sum"), Products=("StockCode", "nunique"),
    Customers=("CustomerID", "nunique"), AvgNetPrice=("NetPrice", "mean"), AvgDiscount=("DiscountPct", "mean"),
).sort_values("Revenue", ascending=False)
cat["Revenue_share"] = cat["Revenue"] / cat["Revenue"].sum()
cat["Cum_share"] = cat["Revenue_share"].cumsum()
print(f"{cat.shape[0]} categories; top 10 = {cat['Revenue_share'].head(10).sum():.1%} of revenue; "
      f"{(cat['Cum_share'] < .8).sum() + 1} categories make 80%")
cat.head(20)

In [ ]:
top = cat.head(20).iloc[::-1].reset_index()
fig = px.bar(top, x="Revenue", y="Category", orientation="h", text=top["Revenue_share"].map("{:.1%}".format),
             title="Top 20 categories by revenue (label = share of total)", hover_data={"Units": ":,.0f", "Products": True, "Customers": True})
fig.update_traces(marker_color=BLUE, textposition="outside", cliponaxis=False)
fig.update_layout(height=650, yaxis_title="", xaxis_title="Revenue")
fig.show()

In [ ]:
# Category tree: revenue share of every category (top 40 shown individually, rest grouped)
tm = cat[["Revenue"]].reset_index()
tm.loc[tm.index >= 40, "Category"] = "Other categories"
tm = tm.groupby("Category", as_index=False)["Revenue"].sum()
fig = px.treemap(tm, path=["Category"], values="Revenue", color="Revenue", color_continuous_scale="Blues",
                 title="Revenue by category (top 40 + other)")
fig.update_traces(hovertemplate="%{label}<br>Revenue: %{value:,.0f}<br>%{percentRoot:.1%} of total<extra></extra>")
fig.update_layout(height=550, coloraxis_showscale=False, margin=dict(l=10, r=10, t=50, b=10))
fig.show()

In [ ]:
# Monthly revenue for the top-6 categories (row-normalised to show seasonality)
top6 = cat.index[:6]
mix = df[df["Category"].isin(top6)].pivot_table(index="Category", columns="Month", values="Revenue", aggfunc="sum").loc[top6]
norm = mix.div(mix.max(axis=1), axis=0)
fig = px.imshow(norm, x=norm.columns.strftime("%b"), y=norm.index, color_continuous_scale="Blues", aspect="auto",
                title="Seasonality of top-6 categories (1 = category's best month)",
                labels=dict(color="Share of peak", x="", y=""))
fig.update_traces(customdata=mix.values, hovertemplate="%{y} · %{x}<br>Revenue: %{customdata:,.0f}<br>%{z:.0%} of peak<extra></extra>",
                  xgap=2, ygap=2)
fig.update_layout(height=380)
fig.show()

## 9. Geography

In [ ]:
city = df.groupby("City").agg(
    Revenue=("Revenue", "sum"), Customers=("CustomerID", "nunique"), Invoices=("InvoiceNo", "nunique")
).sort_values("Revenue", ascending=False)
city["Revenue_share"] = city["Revenue"] / city["Revenue"].sum()
city["Rev_per_customer"] = city["Revenue"] / city["Customers"]
city.head(15)

In [ ]:
top = city.head(15).iloc[::-1]
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, subplot_titles=["Revenue", "Customers"], horizontal_spacing=0.03)
fig.add_bar(y=top.index, x=top["Revenue"], orientation="h", marker_color=BLUE, row=1, col=1,
            customdata=top["Revenue_share"], hovertemplate="%{y}<br>Revenue: %{x:,.0f} (%{customdata:.1%})<extra></extra>")
fig.add_bar(y=top.index, x=top["Customers"], orientation="h", marker_color=BLUE, row=1, col=2,
            hovertemplate="%{y}<br>Customers: %{x}<extra></extra>")
fig.update_layout(title="Top 15 cities", showlegend=False, height=520)
fig.show()

## 10. Invoice prefixes / channels

In [ ]:
pref = df.groupby("Prefix").agg(
    Lines=("Revenue", "size"), Invoices=("InvoiceNo", "nunique"), Customers=("CustomerID", "nunique"),
    Revenue=("Revenue", "sum"), AvgDiscount=("DiscountPct", "mean"),
)
pref["Lines_per_invoice"] = pref["Lines"] / pref["Invoices"]
pref["Revenue_per_invoice"] = pref["Revenue"] / pref["Invoices"]
pref["Revenue_share"] = pref["Revenue"] / pref["Revenue"].sum()
pref = pref.sort_values("Revenue", ascending=False)
pref

In [ ]:
inv = df.groupby(["InvoiceNo", "Prefix"]).agg(Revenue=("Revenue", "sum"), Lines=("Revenue", "size")).reset_index()
fig = px.box(inv, x="Prefix", y="Revenue", category_orders={"Prefix": list(pref.index)}, log_y=True,
             title="Invoice value by prefix (log scale)", labels={"Revenue": "Revenue per invoice"})
fig.update_traces(marker_color=BLUE, line_color=BLUE, marker_size=3)
fig.update_layout(height=420)
fig.show()

## 11. Correlations

In [ ]:
corr = cust[["Revenue", "Units", "Invoices", "Products", "AvgInvoice"]].corr(method="spearman")
fig = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                title="Customer-level Spearman correlations")
fig.update_traces(xgap=2, ygap=2)
fig.update_layout(height=480, width=560)
fig.show()

In [ ]:
c = cust.reset_index()
fig = px.scatter(c, x="Invoices", y="Revenue", log_x=True, log_y=True, hover_name="CustomerID",
                 hover_data={"City": True, "Products": True, "Revenue": ":,.0f"},
                 title="Customer frequency vs revenue (log-log)")
fig.update_traces(marker=dict(color=BLUE, size=7, opacity=.55, line=dict(color="white", width=0.5)))
fig.update_layout(height=480)
fig.show()

## 12. Key findings

In [ ]:
best_m = monthly["Revenue"].idxmax(); worst_m = monthly["Revenue"].idxmin()
print(f"• {len(df):,} clean lines, {df['InvoiceNo'].nunique():,} invoices, {df['CustomerID'].nunique():,} customers, "
      f"{df['StockCode'].nunique():,} products, {df['Category'].nunique()} categories, {df['City'].nunique()} cities.")
print(f"• Net revenue {df['Revenue'].sum():,.0f}; effective trade discount {1 - df['Revenue'].sum()/df['ListValue'].sum():.1%} off list price.")
print(f"• Best month {best_m:%b %Y} ({monthly.loc[best_m,'Revenue']:,.0f}); weakest {worst_m:%b %Y} ({monthly.loc[worst_m,'Revenue']:,.0f}).")
print(f"• Top 20% of customers drive {top20:.1%} of revenue; {n80} customers make up 80%.")
print(f"• Top category '{cat.index[0]}' = {cat['Revenue_share'].iat[0]:.1%} of revenue; top 10 categories = {cat['Revenue_share'].head(10).sum():.1%}.")
print(f"• {city.index[0]} alone = {city['Revenue_share'].iat[0]:.1%} of revenue; top 5 cities = {city['Revenue_share'].head(5).sum():.1%}.")
print(f"• Prefixes BAD + BSR carry {pref.loc[['BAD','BSR'],'Revenue_share'].sum():.1%} of revenue.")
print(f"• Data quality: {n_dup:,} exact duplicate lines, {issues['City missing']:,} missing cities, a handful of zero/negative price or quantity lines (dropped).")

### Next steps for the dashboard
- Build RFM / K-Means customer segmentation on the customer table above.
- Market-basket analysis (association rules) on `InvoiceNo × Category`.
- Monthly revenue forecasting per top category.
- Streamlit + Plotly dashboard with filters for date range, city, category and prefix.